# RNA-seq: Full Pipeline (our data + JDM lab data → merged, normalised, batch-corrected)

**What this notebook does, end to end.**

1. Tidy our RNA-seq files and attach patient demographics.
2. Keep **one whole-blood, pre-treatment sample per patient**.
3. Compute RPKM and TPM.
4. Load the **JDM lab's** processed files and work out who their samples are.
5. Check that our RPKM reproduces theirs on shared samples.
6. **Merge** both datasets and quantile-normalise jointly.
7. Remove the dataset (batch) effect while keeping disease differences.
8. Write a processing log generated from the run itself.

**How to read it.** Each section has a 🎯 **Goal**, a 💡 **Why**, your code unchanged, and a 👀 **Checkpoint** (a table or plot) so you can see the result before moving on.

**Roadmap**

| Part | Step | Output in memory |
|---|---|---|
| 1 | Configuration & helpers | paths, rules, `qnorm`, `shape` |
| 2 | Patient metadata | `md`, lookups |
| 3 | Load & filter our RNA files | `rmeta`, `counts`, `final_meta` |
| 4 | Gene lengths → RPKM / TPM | `rpkm`, `tpm`, `L` |
| 5 | JDM lab files, key sheet, demographics | `rp`, `qn`, `key` |
| 6 | Cross-check against their numbers | `pairs`, `cors` |
| 7 | Merge + joint quantile normalisation | `QN`, `merged_meta` |
| 8 | Processing log | `PROCESSING_LOG.md` |
| 9 | Batch correction | `merged_log2_RPKM_QN_batchcorrected.csv` |

<div class="alert alert-info">
<b>Design principle: filter by flags, never by deletion.</b> Every sample gets a <code>status</code> explaining why it was kept or dropped, and <code>rnaseq_samples_ALL_with_flags.csv</code> preserves everything, so any exclusion can be audited later.
</div>

<div class="alert alert-warning">
<b>Privacy.</b> Birth date is read only to compute age, then dropped. Printed output never shows real IDs, MRNs or birth dates; IDs are masked to their format (<code>JDM_181 → AAA_999</code>). All outputs are CSV.
</div>

<div class="alert alert-danger">
<b>First run is slow.</b> The xlsx→csv conversion and the GTF download both happen once and are cached afterwards.
</div>

---
# Part 1 · Configuration & helpers

🎯 **Goal:** every path, rule and cohort decision in one place, plus the small tools used throughout.

💡 **Why:** the scientific *decisions* are variables, not buried logic:

| Setting | What it decides |
|---|---|
| `KEEP_ONE_PER_PATIENT` | `"earliest"` ≈ first-visit design; `"deepest"` favours sequencing depth |
| `REQUIRE_PRE_FOR_DX` | for CANDLE / SAVI the data *has* PRE/POST labels, so an unlabelled sample is not trusted to be pre-treatment |
| `BLOOD_VALUES` | **exact** match, so "PBMC" can never slip through as "blood" |
| `LIB_CONVENTION` | which library size to use for RPKM when merging (see Part 7) |
| `DROP_PEOPLE_ALREADY_IN_OURS` | avoids counting the same person twice across the two datasets |
| `DX_MAP` | harmonises diagnosis spellings between files |

### 1.1 Imports and settings

In [ ]:
# =============================================================================
# RNA-seq, ONE CELL (everything, all outputs CSV).
#   tidy + annotate our RNA files -> whole blood only -> drop post-treatment ->
#   one sample per patient -> RPKM / TPM -> JDM-lab files: key sheet, demographics
#   (multi-step matching, age only, never birth date) -> compare -> merge both
#   datasets and quantile-normalise -> processing log -> copy to the mount.
# Paste into ONE notebook cell and run. First run is slow (xlsx->csv conversion,
# GTF download); both are cached afterwards.
# Printing never shows real IDs, MRNs or birth dates (masked formats and counts).
# =============================================================================
import os, re, glob, gzip, shutil, datetime, platform, warnings, urllib.request
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ---- configuration ----------------------------------------------------------
MNT = "/mnt/file-systems/"
RNA_FILES = [
    MNT + "HC_blood_all_samples.xlsx",
    MNT + "Goldbach-Mansky_compiled_RNA-Seq_GeneCounts_CANDLE_AND_SAVI_SUBSET_2026_09_22.xlsx",
]
METADATA_PATH = MNT + "MetadataPatients_NoPHI.xlsx"
METADATA_EXTRA = []        # optional extra metadata sheets with the same headers
JDM_RPKM = MNT + "4b_rpkm.csv"
JDM_QN   = MNT + "4a_log2QN.csv"
JDM_KEY  = MNT + "jdm-HC RNASEQ sample ID.csv"
OUT_DIR   = "/mnt/file-systems/rnaseq_out"   # results are written here directly
CACHE_DIR = "/home/jovyan/rnaseq_cache"      # xlsx->csv cache stays local
# at the end, OUT_DIR is copied to the first of these that exists and is writable
COPY_TO = ["/mnt/file-systems", "/mnt/session_data/filesystems",
           "/home/jovyan/session_data/filesystems"]

KEEP_ONE_PER_PATIENT = "earliest"        # or "deepest" (most total counts)
REQUIRE_PRE_FOR_DX = r"CANDLE|SAVI"      # these cohorts have PRE/POST labels: insist on PRE
BLOOD_VALUES = {"blood", "whole blood", "wholeblood", "wb"}     # exact match; PBMC never passes
LIB_CONVENTION = "their_gene_set"        # RPKM library size: "their_gene_set" or "all_genes"
DROP_PEOPLE_ALREADY_IN_OURS = True       # JDM-lab samples from a person already in our kept set are dropped
GTF_PATH = ""                            # blank = auto-find, then try download
GTF_URL = ("https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/"
           "release_19/gencode.v19.annotation.gtf.gz")
DX_MAP = {"hc whole blood": "HC",
          "relative-hc": "HC",
          "undifferentiated interferonopathy": "Undifferentiated Interferonopathy"}
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
assert os.access(OUT_DIR, os.W_OK), f"{OUT_DIR} is not writable"

### 1.2 Helper functions

💡 **Why each one exists:**

| Helper | Reason |
|---|---|
| `normalize_id` | `O-6033` / `o6033` / `O6033` compare equal; strips Excel's `.0`; maps RNA's `CHLD0047` to metadata's `LD0047` |
| `zs` | drops zero-padding after a letter (`JDM0181 → JDM181`), because the two systems pad differently |
| `shape` | **privacy**: masks an ID to its format (`JDM_181 → AAA_999`) so you can diagnose matching problems without printing real IDs |
| `read_table_cached` | large `.xlsx` files take minutes to read; converts to CSV once, then reads the cache. Everything is text so `0047` keeps its zeros |
| `qnorm` | **quantile normalisation**: forces every sample to have the *same distribution* by replacing each value with the mean of the same-rank values across samples |
| `finfo` | file name, size and modified date, for the provenance log |

In [ ]:
# ---- helpers ----------------------------------------------------------------
def normalize_id(x):
    s = "" if x is None else str(x).strip()
    if s.lower() in ("nan", "none", "nat", ""):
        return ""
    s = re.sub(r"\.0$", "", s)
    s = s.upper().replace("-", "").replace(" ", "").replace("_", "")
    if s.startswith("CHLD"):                 # CHLD0047 (RNA files) == LD0047 (metadata)
        s = s[2:]
    return s


def zs(s):
    """drop zero-padding that follows a letter: JDM0181 -> JDM181"""
    return re.sub(r"(?<=[A-Z])0+(?=\d)", "", s)


def shape(x):
    """mask an ID so only its format shows: 'JDM_181' -> 'AAA_999'"""
    return re.sub(r"\d", "9", re.sub(r"[A-Za-z]", "A", str(x)))


def norm_colname(c):
    return re.sub(r"[^a-z0-9]", "", str(c).lower())


DATE_MIN, DATE_MAX = pd.Timestamp("2008-01-01"), pd.Timestamp.today().normalize()


def read_table_cached(path):
    """xlsx -> csv once (cached in CACHE_DIR), then always read the csv."""
    if path.lower().endswith(".csv"):
        return pd.read_csv(path, dtype=object, keep_default_na=False, low_memory=False)
    csv_path = os.path.join(CACHE_DIR, os.path.splitext(os.path.basename(path))[0] + ".csv")
    if (not os.path.exists(csv_path)
            or os.path.getmtime(csv_path) < os.path.getmtime(path)):
        print(f"converting {os.path.basename(path)} -> csv (one time) ...")
        pd.read_excel(path, dtype=object).to_csv(csv_path, index=False)
    return pd.read_csv(csv_path, dtype=object, keep_default_na=False, low_memory=False)


def qnorm(df):
    """Quantile normalisation across the columns of df (no NaN allowed)."""
    mean_sorted = np.sort(df.values, axis=0).mean(axis=1)
    ranks = df.rank(method="first").astype(int).values - 1
    return pd.DataFrame(mean_sorted[ranks], index=df.index, columns=df.columns)


def finfo(p):
    return (f"{os.path.basename(p)}  ({os.path.getsize(p)/1e6:.1f} MB, modified "
            f"{datetime.datetime.fromtimestamp(os.path.getmtime(p)):%Y-%m-%d})")

---
# Part 2 · Patient metadata

🎯 **Goal:** load the master patient sheet and build fast lookups from it.

💡 **Why:**

- The workbook may have several sheets; the one with the most *enroll / diagnos / initial* columns is chosen automatically.
- `METADATA_EXTRA` appends more patients; the main sheet is first, so it wins on duplicate IDs.
- Column names are found **loosely**, because headers drift between versions.
- Lookups built here: **ID → row** (both Enrollment and Alternate ID), a **zero-padding-insensitive** version of the same, **unique initials → row/ID** (shared initials are never auto-resolved: a wrong patient is worse than a blank one), and **IDs hidden in the Notes column** (a weak last-resort for matching).

In [ ]:
# =============================================================================
# A. patient metadata
# =============================================================================
sheets = pd.read_excel(METADATA_PATH, sheet_name=None, dtype=str)
md = max(sheets.values(), key=lambda d: sum(bool(re.search(r"enroll|diagnos|initial", str(c), re.I))
                                            for c in d.columns))
md.columns = [str(c).strip() for c in md.columns]
md = md.dropna(how="all").reset_index(drop=True)

for p in METADATA_EXTRA:                     # main sheet first, so it wins on duplicate IDs
    ex = (pd.read_excel(p, dtype=str) if p.lower().endswith((".xlsx", ".xls"))
          else pd.read_csv(p, dtype=str))
    ex.columns = [str(c).strip() for c in ex.columns]
    nm = {norm_colname(c): c for c in md.columns}
    ex = ex.rename(columns={c: nm[norm_colname(c)] for c in ex.columns if norm_colname(c) in nm})
    md = pd.concat([md, ex.dropna(how="all").reindex(columns=md.columns)], ignore_index=True)
    print(f"extra metadata: {p} (+{len(ex)} rows)")


def fc(*names):
    nm = {norm_colname(c): c for c in md.columns}
    return next((nm[norm_colname(n)] for n in names if norm_colname(n) in nm), None)


COL = {"enroll": fc("Enrollment ID"), "alt": fc("Alternate Enrollment ID"),
       "initials": fc("Initials"), "dob": fc("Birth Date", "DOB"), "sex": fc("Sex", "Gender"),
       "race": fc("Race"), "ethnicity": fc("Ethnicity"), "status": fc("Study Status"),
       "dx1": fc("Diagnosis1"), "dx2": fc("Diagnosis2"), "note": fc("Note", "Notes")}
print("metadata:", len(md), "rows; columns not found:", [k for k, v in COL.items() if v is None])

meta_lookup, zs_lookup, by_ini, ini_rows, notes_norm = {}, {}, {}, {}, {}
for i, r in md.iterrows():
    for kc in (COL["enroll"], COL["alt"]):
        k = normalize_id(r.get(kc)) if kc else ""
        if k:
            meta_lookup.setdefault(k, i)
            zs_lookup.setdefault(zs(k), i)
    ini = str(r.get(COL["initials"]) or "").strip().upper() if COL["initials"] else ""
    eid = normalize_id(r.get(COL["enroll"])) if COL["enroll"] else ""
    if ini and eid:
        by_ini.setdefault(ini, set()).add(eid)
    if ini:
        ini_rows.setdefault(ini, set()).add(i)
    if COL["note"]:
        notes_norm[i] = normalize_id(r.get(COL["note"]))
AUTO_INITIALS_MAP = {k: next(iter(v)) for k, v in by_ini.items() if len(v) == 1}
INI_UNIQUE_ROW = {k: next(iter(v)) for k, v in ini_rows.items() if len(v) == 1}


def meta_value(i, key):
    c = COL.get(key)
    if i < 0 or not c:
        return ""
    v = md.at[i, c]
    return "" if pd.isna(v) else str(v).strip()

---
# Part 3 · Our RNA files: load, filter, one per patient

🎯 **Goal:** a cohort of **one whole-blood, pre-treatment sample per patient**, with demographics.

The sample count narrows like a funnel; the checkpoint at the end of this part plots it.

### 3.1 Load: split metadata rows from count rows

💡 **Why the reshaping:** in the compiled spreadsheets, rows labelled `Raw_Count` are genes, but **every other row is sample metadata** (diagnosis, treatment...) stored *as if it were a gene*. The loader splits them and **transposes** the metadata so each sample becomes a row. It fails with a clear message if required columns are missing, and **warns about any sample whose counts are entirely non-numeric**, a common symptom of a corrupted spreadsheet column.

Files are combined on `Gene_ID` with an **inner join** (only genes present in *every* file), so all samples share one gene set. Repeated sample names across files raise an error rather than silently colliding.

In [ ]:
# =============================================================================
# B. our RNA files: split metadata rows from count rows, samples become rows
# =============================================================================
def load_rna(path):
    raw = read_table_cached(path)
    raw.columns = [str(c).strip() for c in raw.columns]
    need = {"Gene_ID", "Gene_Name", "Stat"}
    if not need <= set(raw.columns):
        raise ValueError(f"{os.path.basename(path)} lacks {need - set(raw.columns)}; "
                         f"first columns are {list(raw.columns)[:6]}")
    samples = [c for c in raw.columns if c not in need]
    is_count = raw["Stat"].astype(str).str.strip().eq("Raw_Count")
    print(f"{os.path.basename(path)[:60]}: {len(samples)} samples, {int(is_count.sum())} genes")
    meta = raw.loc[~is_count, ["Stat"] + samples].set_index("Stat").T
    meta.columns = [str(c).strip() for c in meta.columns]
    meta.index.name = "sample_id"
    cnt = raw.loc[is_count, ["Gene_ID", "Gene_Name"] + samples].copy()
    cnt[samples] = cnt[samples].apply(pd.to_numeric, errors="coerce")
    for s in samples:
        if cnt[s].isna().all():
            print(f"WARNING: no numeric counts in sample {s} "
                  f"(raw examples: {raw.loc[is_count, s].head(3).tolist()})")
    return meta.reset_index(), cnt


metas, cnts = [], []
for p in RNA_FILES:
    m, c = load_rna(p)
    m["source_file"] = os.path.basename(p)
    metas.append(m)
    cnts.append(c)

gene_names = cnts[0].drop_duplicates("Gene_ID").set_index("Gene_ID")["Gene_Name"]
mats = [c.drop(columns="Gene_Name").groupby("Gene_ID").sum(min_count=1) for c in cnts]
counts = pd.concat(mats, axis=1, join="inner")
print(f"genes in every file: {len(counts)}  (per file: {[len(m) for m in mats]})")
dup = counts.columns[counts.columns.duplicated()]
if len(dup):
    raise ValueError(f"sample names repeated across files: {list(dup)[:5]}")

### 3.2 Standardise sample metadata and flag empty samples

💡 **Why:** the two files name the same field differently, so a rename map gives each field one canonical name (missing ones are created blank so nothing raises `KeyError` later). Library size (`total_counts`) is computed now because it drives tie-breaking and RPKM. Samples with **no counts at all** are flagged `has_counts = False` and excluded later, with the reason recorded.

In [ ]:
rn = {"enrollmentid": "enrollment_id_rna", "metadatainitials": "initials_rna",
      "samplesource": "sample_source", "diagnosis": "diagnosis_rna", "treatment": "treatment",
      "sequencingfacility": "sequencing_facility", "sequencingcohort": "sequencing_cohort",
      "gender": "gender_rna", "rnasamplecollectiondate": "rna_collection_date"}
rmeta = pd.concat(metas, ignore_index=True)
rmeta = rmeta.rename(columns=lambda c: rn.get(norm_colname(c), c))
for c in rn.values():
    if c not in rmeta.columns:
        rmeta[c] = ""
rmeta = rmeta.fillna("").replace({"-": "", "NaN": "", "nan": ""})
rmeta["total_counts"] = rmeta["sample_id"].map(counts.sum(min_count=1))
rmeta["has_counts"] = rmeta["total_counts"].fillna(0) > 0
print(f"\nsamples with no counts at all (excluded below): {int((~rmeta['has_counts']).sum())}",
      rmeta.loc[~rmeta["has_counts"], "sample_id"].tolist())
rmeta["dx_clean"] = rmeta["diagnosis_rna"].map(
    lambda s: DX_MAP.get(str(s).strip().lower(), str(s).strip()))

### 3.3 Whole blood only

💡 **Why two layers of defence:** a PBMC or sorted-cell sample has a very different transcriptome (no neutrophils, different interferon signal), so mixing it with whole blood would create a fake effect.

1. **Label check:** `sample_source` must exactly equal one of `BLOOD_VALUES`.
2. **Sneaky check:** even if labelled "blood", the sample name / treatment / cohort text is searched for cell-fraction words (`pbmc`, `cd4`, `sorted`...). Anything caught is **excluded and printed**.

The crosstab shows every `sample_source` spelling per file, so a new variant that should count as blood is easy to spot.

In [ ]:
# ---- whole blood only (no PBMC, no sorted cell fractions) -------------------
src = rmeta["sample_source"].astype(str).str.strip().str.lower()
print("\nsample source values (per file):")
print(pd.crosstab(rmeta["source_file"].str[:40], src).to_string())
NOT_WHOLE_BLOOD = re.compile(r"pbmc|mononuclear|monocyte|neutrophil|lymphocyte|"
                             r"t[\s_-]?cell|b[\s_-]?cell|nk[\s_-]?cell|cd\d|sorted|isolat", re.I)
blob = (rmeta["sample_id"].astype(str) + " " + src + " " + rmeta["treatment"].astype(str)
        + " " + rmeta["sequencing_cohort"].astype(str))
sneaky = src.isin(BLOOD_VALUES) & blob.str.contains(NOT_WHOLE_BLOOD)
print(f"\nlabelled blood but name suggests a cell fraction: {int(sneaky.sum())}")
if sneaky.any():
    print(rmeta.loc[sneaky, ["sample_id", "sample_source"]].head(20).to_string())
rmeta["is_blood"] = src.isin(BLOOD_VALUES) & ~sneaky

### 3.4 Dates (for age and ordering only)

💡 **Why a priority order:** a real `rna_collection_date` is preferred. If missing, a date is mined from the sample name (e.g. `121003_121016`). That stamp is a **sequencing batch**, shared by PRE and POST samples, so it is only a rough proxy for the draw date. Its provenance is recorded in `age_date_source` so approximate ages can be filtered later. Dates outside 2008–today are rejected as typos.

In [ ]:
# ---- date (for age and ordering only) ---------------------------------------
# The stamp in sample names (e.g. 121003_121016) is a sequencing batch, shared by
# PRE and POST samples, so it is only a rough proxy for the draw date.
def date_from_name(s):
    m = re.search(r"(?<!\d)(\d{6})_\d{6}(?!\d)", s) or re.search(r"(?<!\d)(\d{6})(?!\d)", s)
    if not m:
        return pd.NaT
    try:
        d = pd.to_datetime(m.group(1), format="%y%m%d")
    except ValueError:
        return pd.NaT
    return d if DATE_MIN <= d <= DATE_MAX else pd.NaT


coll = pd.to_datetime(rmeta["rna_collection_date"], errors="coerce")
coll = coll.where((coll >= DATE_MIN) & (coll <= DATE_MAX))
from_name = rmeta["sample_id"].map(date_from_name)
rmeta["_date"] = coll.fillna(from_name)
rmeta["age_date_source"] = np.where(coll.notna(), "rna_collection_date",
                            np.where(from_name.notna(), "sample_name (approx)", "none"))

### 3.5 Join patient metadata

💡 **Why a careful ladder:** the RNA files and the patient sheet were never designed to join. In order of trust:

1. **Enrollment ID** exact (`enrollment_id`).
2. **HC fix:** bare `1126` is `R1126` in metadata (`R_prefix_id`).
3. **Initials, only when safe:** only if the RNA ID is blank or itself initials-derived (`HC-MF`). A real-looking ID the metadata lacks returns `id_not_in_metadata`, so it **never guesses**.
4. **Diagnosis sanity check** on initials matches: a conflict returns `initials_conflict` and stays unmatched.

`demographics_trusted` is true only for the two strong ID methods, so you can filter out anything resting on initials.

In [ ]:
# ---- join patient metadata --------------------------------------------------
HEALTHY = re.compile(r"^hc$|health|control", re.I)
MYOSITIS = re.compile(r"myosit|dermatomyos|jdm", re.I)


def match_meta(r):
    k = normalize_id(r["enrollment_id_rna"])
    ini = str(r["initials_rna"]).strip().upper()
    ini_n = normalize_id(ini)
    if k and k in meta_lookup:
        return meta_lookup[k], "enrollment_id"
    if k.isdigit() and ("R" + k) in meta_lookup:           # HC '1126' is 'R1126' in the metadata
        return meta_lookup["R" + k], "R_prefix_id"
    # only fall back to initials when the RNA ID is blank or is itself initials-derived (HC-MF, BG-SAF)
    if not (k == "" or (ini_n and k.endswith(ini_n))):
        return -1, "id_not_in_metadata"                    # a real-looking ID the metadata lacks: don't guess
    eid = AUTO_INITIALS_MAP.get(ini)                       # unique initials only, as in NanoString
    if eid and eid in meta_lookup:
        i = meta_lookup[eid]
        dx_md, dx_rna = meta_value(i, "dx1"), str(r["diagnosis_rna"])
        same = (norm_colname(dx_md) == norm_colname(dx_rna)
                or (HEALTHY.search(dx_md) and HEALTHY.search(dx_rna)))
        compat = (re.search(r"interferon|aicardi|\bags\b", dx_md, re.I)
                  and re.search(r"\bags\b|aicardi", dx_rna, re.I))
        if dx_md == "" or same:
            return i, "initials"
        if compat:
            return i, "initials_dx_compat"
        return -1, "initials_conflict"
    return -1, "none"


mm = rmeta.apply(lambda r: pd.Series(match_meta(r), index=["meta_row", "meta_join"]), axis=1)
rmeta = pd.concat([rmeta, mm], axis=1)
extra_cols = {}
for f, name in [("sex", "sex"), ("race", "race"), ("ethnicity", "ethnicity"),
                ("status", "study_status"), ("dx1", "diagnosis1"), ("dx2", "diagnosis2"),
                ("note", "metadata_note"), ("dob", "_dob"), ("enroll", "_md_enroll")]:
    extra_cols[name] = rmeta["meta_row"].map(lambda i, f=f: meta_value(i, f))
rmeta = pd.concat([rmeta, pd.DataFrame(extra_cols)], axis=1)
rmeta["demographics_trusted"] = rmeta["meta_join"].isin(["enrollment_id", "R_prefix_id"])
print("\nmetadata join:")
print(rmeta["meta_join"].value_counts().to_string())

dob = pd.to_datetime(rmeta["_dob"], errors="coerce")
rmeta["age_yrs"] = ((rmeta["_date"] - dob).dt.days / 365.25).round(1)
rmeta.loc[(rmeta["age_yrs"] < 0) | (rmeta["age_yrs"] > 120), "age_yrs"] = np.nan

### 3.6 Treatment status

💡 **Why this is the most important filter:** drugs (e.g. JAK inhibitors) can suppress an interferon signature almost completely. A post-treatment sample then looks *healthy* and quietly dilutes the disease cohort.

A pattern recognises "pre" wording (`pre`, `baseline`, `untreated`, `naive`, `screening`, `day 0`...). **Any other non-blank text is treated as post-treatment**, the cautious default. Blank stays `unlabelled`; its fate depends on diagnosis (next step).

In [ ]:
# ---- treatment status -------------------------------------------------------
PRE_TX = re.compile(r"pre|baseline|bsl|untreat|naive|none|before|screen|off[\s-]?(?:tx|treat)|^v?0$|^day\s*0$", re.I)
tx = rmeta["treatment"].astype(str).str.strip()
is_pre = (tx != "") & tx.str.contains(PRE_TX)
rmeta["is_post_tx"] = (tx != "") & ~is_pre
rmeta["tx_status"] = np.where(rmeta["is_post_tx"], "post", np.where(is_pre, "pre", "unlabelled"))
print("\ntreatment status (per file):")
print(pd.crosstab(rmeta["source_file"].str[:40], rmeta["tx_status"]).to_string())

### 👀 Checkpoint 3a: did the pre/post regex call things correctly?

**Read this table.** The regex is a heuristic and a wrong call changes who is in your cohort. It shows each raw `treatment` string and the status it received. Fix `PRE_TX` if anything is misclassified.

In [ ]:
import matplotlib.pyplot as plt

display(rmeta.assign(treatment=rmeta["treatment"].replace("", "(blank)"))
        .groupby(["tx_status", "treatment"]).size().rename("n")
        .reset_index().sort_values(["tx_status", "n"], ascending=[True, False]).head(40))

### 3.7 One sample per patient

💡 **Why one per patient:** patients with many samples would dominate, and repeated measures from one person aren't independent, which violates the assumptions of standard tests.

**Eligibility:** whole blood, has counts, **not** post-treatment, and a patient identity exists. For CANDLE/SAVI an **unlabelled** sample is excluded rather than assumed pre-treatment.

**Choosing among a patient's samples:** explicit `PRE` beats unlabelled; then *earliest* date (or deepest library if `"deepest"`); library size breaks ties.

**`patient_key`** is the best identity available (metadata enrollment ID → RNA file's ID → initials), so even unmatched samples de-duplicate against each other.

`status` then records **exactly why every sample was kept or dropped**, in priority order.

In [ ]:
# ---- one sample per patient: blood, not post-treatment ----------------------
rmeta["patient_key"] = [normalize_id(a) or normalize_id(b) or normalize_id(c)
    for a, b, c in zip(rmeta["_md_enroll"], rmeta["enrollment_id_rna"], rmeta["initials_rna"])]
need_pre = rmeta["dx_clean"].astype(str).str.contains(REQUIRE_PRE_FOR_DX, case=False, regex=True)
base = (rmeta["is_blood"] & rmeta["has_counts"] & ~rmeta["is_post_tx"]
        & (rmeta["patient_key"] != ""))
print(f"\nexcluded for lacking a PRE label ({REQUIRE_PRE_FOR_DX}):", int((base & need_pre & ~is_pre).sum()))
elig = base & (~need_pre | is_pre)
b = rmeta[elig].copy()
b["_pre_rank"] = (b["tx_status"] != "pre").astype(int)       # explicit PRE beats unlabelled
if KEEP_ONE_PER_PATIENT == "earliest":
    b = b.sort_values(["patient_key", "_pre_rank", "_date", "total_counts"],
                      ascending=[True, True, True, False], na_position="last")
else:
    b = b.sort_values(["patient_key", "_pre_rank", "total_counts"], ascending=[True, True, False])
rmeta["kept"] = rmeta["sample_id"].isin(b.drop_duplicates("patient_key")["sample_id"])
final_meta = rmeta[rmeta["kept"]].copy()

# why every sample was kept or dropped
rmeta["status"] = np.select(
    [~rmeta["has_counts"], ~rmeta["is_blood"], rmeta["is_post_tx"], rmeta["patient_key"] == "",
     need_pre & ~is_pre, ~rmeta["kept"]],
    ["no counts in file", "not whole blood", "post-treatment", "no patient ID",
     "CANDLE/SAVI without PRE label", "other sample from same patient"],
    default="kept")

print(f"\nsamples: {len(rmeta)} -> {int(rmeta['is_blood'].sum())} blood "
      f"-> {int(elig.sum())} eligible -> {len(final_meta)} one-per-patient")
print("age known:", int(final_meta["age_yrs"].notna().sum()), "of", len(final_meta))
for title, col in [("treatment status", "tx_status"), ("metadata join", "meta_join"),
                   ("age date source", "age_date_source")]:
    print(f"\nkept samples by {title}:")
    print(pd.crosstab(final_meta["dx_clean"].replace("", "(blank)"), final_meta[col]).to_string())
ini_ct = final_meta.assign(ini=final_meta["initials_rna"].str.upper()).groupby("ini")["patient_key"].agg(list)
print("\nsame initials under more than one patient key (possible duplicate person):")
print(ini_ct[ini_ct.str.len() > 1].to_string() if (ini_ct.str.len() > 1).any() else "  none")

### 👀 Checkpoint 3b: the attrition funnel

**Left:** every sample's final `status`, so you can see which rule removed the most. **Right:** the final cohort by diagnosis, coloured by how its demographics were matched. A lot of non-green here means a lot of age/sex values rest on weak matches.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
rmeta["status"].value_counts().plot.barh(ax=ax[0], color="#4C78A8")
ax[0].invert_yaxis(); ax[0].set_title(f"Fate of all {len(rmeta)} samples")
pd.crosstab(final_meta["dx_clean"].replace("", "(blank)"), final_meta["meta_join"]
            ).plot.barh(stacked=True, ax=ax[1])
ax[1].invert_yaxis(); ax[1].set_title("Kept cohort by diagnosis and match method"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()

---
# Part 4 · Gene lengths → RPKM and TPM

🎯 **Goal:** length-normalised expression, because raw counts aren't comparable *between genes* (long genes collect more reads) or *between samples* (deeper libraries collect more).

💡 **Why two measures:**

| Measure | Formula | Properties |
|---|---|---|
| **RPKM** | counts ÷ length(kb) ÷ library(millions) | depth then length; column sums differ between samples |
| **TPM** | (counts ÷ length) ÷ Σ(counts ÷ length) × 10⁶ | every sample sums to exactly 1,000,000; better for cross-sample comparison |

**Gene length** = the *non-overlapping exon union* per gene from a GENCODE GTF, so exons shared by several transcripts aren't double-counted. Order of search: cached `gene_lengths.csv` → any `.gtf` on the mounts → download GENCODE v19. Version suffixes (`ENSG….5`) are stripped when IDs don't match, and the **match rate is printed**. If it's low, the GTF release doesn't match your counts.

Raw counts are written *before* this step can fail, and a missing GTF raises a clear error telling you what to do.

### 4.1 Gene lengths (cached)

In [ ]:
# =============================================================================
# C. gene lengths -> RPKM and TPM for the kept samples
# =============================================================================
def gtf_gene_lengths(path):
    op = gzip.open if path.endswith(".gz") else open
    rows = []
    with op(path, "rt") as fh:
        for line in fh:
            if line[0] == "#":
                continue
            f = line.split("\t")
            if len(f) > 8 and f[2] == "exon":
                rows.append((re.search(r'gene_id "([^"]+)"', f[8]).group(1), int(f[3]), int(f[4])))
    ex = pd.DataFrame(rows, columns=["g", "s", "e"]).sort_values(["g", "s"])
    prev = ex.groupby("g")["e"].cummax().groupby(ex["g"]).shift().fillna(0)
    return ((ex["e"] - np.maximum(ex["s"] - 1, prev)).clip(lower=0)).groupby(ex["g"]).sum()


len_csv = os.path.join(OUT_DIR, "gene_lengths.csv")
lengths = pd.read_csv(len_csv, index_col=0).iloc[:, 0] if os.path.exists(len_csv) else None
if lengths is None:
    gtf = GTF_PATH or next(iter(sum((glob.glob(f"{r}/**/*.gtf*", recursive=True)
            for r in ["/mnt/file-systems", "/mnt/session_data", "/home/jovyan"]), [])), "")
    if not gtf:
        try:
            gtf = "/home/jovyan/gencode.v19.annotation.gtf.gz"
            print("downloading", GTF_URL)
            urllib.request.urlretrieve(GTF_URL, gtf)
        except Exception as e:                                   # noqa: BLE001
            print("could not download GTF:", e)
            gtf = ""
    if gtf:
        lengths = gtf_gene_lengths(gtf)
        lengths.to_csv(len_csv, header=["length_bp"])

### 4.2 Write raw counts, compute RPKM / TPM

In [ ]:
cc = counts[list(final_meta["sample_id"])]
cc_out = cc.copy()
cc_out.insert(0, "Gene_Name", cc_out.index.map(gene_names))
cc_out.to_csv(os.path.join(OUT_DIR, "rnaseq_counts_raw.csv"))

if lengths is None:
    raise RuntimeError("No gene lengths available. Raw counts were written. Put a .gtf(.gz) "
                       "on the mount and set GTF_PATH, then re-run.")

versioned = counts.index.isin(lengths.index).mean() > 0.5
lkp, gkey = lengths.copy(), counts.index
if not versioned:
    lkp.index = lkp.index.str.split(".").str[0]
    lkp = lkp[~lkp.index.duplicated()]
    gkey = counts.index.str.split(".").str[0]
L = pd.Series(lkp.reindex(gkey).values / 1000.0, index=counts.index)
ok = L.notna() & (L > 0)
print(f"\ngene length found for {L.notna().mean():.1%} of {len(L)} genes")

lib = cc.sum()                                   # full library size, all genes
c2 = cc[ok]
rpkm = c2.div(L[ok], axis=0).div(lib / 1e6, axis=1)
rate = c2.div(L[ok], axis=0)
tpm = rate.div(rate.sum(), axis=1) * 1e6
for nm_, mat in (("rpkm", rpkm), ("tpm", tpm)):
    o = mat.copy()
    o.insert(0, "Gene_Name", o.index.map(gene_names))
    o.to_csv(os.path.join(OUT_DIR, f"rnaseq_{nm_}.csv"))

### 4.3 Sample sheet, weak-match list, and a samples × genes master

💡 **Why these files:**

| File | Use |
|---|---|
| `rnaseq_samples.csv` | the analysis cohort, one row per kept sample, no DOB |
| `unmatched_or_weak_metadata.csv` | kept samples whose demographics are **not** trusted: the to-do list |
| `rnaseq_samples_ALL_with_flags.csv` | **everything**, with `status`, for auditing |
| `rnaseq_master_samples_x_genes_raw.csv` | metadata + raw counts in one table (columns named `Symbol\|EnsemblID`, so neither is lost) |

In [ ]:
# ---- sample sheet, unmatched list, one-row-per-sample master ----------------
cols = ["sample_id", "patient_key", "enrollment_id_rna", "initials_rna", "meta_join",
        "demographics_trusted", "sample_source", "treatment", "tx_status", "diagnosis_rna",
        "dx_clean", "diagnosis1", "diagnosis2", "study_status", "sex", "race", "ethnicity",
        "age_yrs", "age_date_source", "sequencing_facility", "sequencing_cohort",
        "total_counts", "source_file"]
final_meta[cols].to_csv(os.path.join(OUT_DIR, "rnaseq_samples.csv"), index=False)
final_meta.loc[~final_meta["demographics_trusted"],
               ["sample_id", "enrollment_id_rna", "initials_rna", "meta_join", "dx_clean", "source_file"]
               ].to_csv(os.path.join(OUT_DIR, "unmatched_or_weak_metadata.csv"), index=False)
rmeta.drop(columns=["_dob", "_date", "_md_enroll", "meta_row"]).to_csv(
    os.path.join(OUT_DIR, "rnaseq_samples_ALL_with_flags.csv"), index=False)
mat_wide = cc.T.copy()
mat_wide.columns = [f"{gene_names.get(g, g)}|{g}" for g in mat_wide.columns]
master = pd.concat([final_meta[cols].set_index("sample_id"), mat_wide], axis=1).reset_index()
master.to_csv(os.path.join(OUT_DIR, "rnaseq_master_samples_x_genes_raw.csv"), index=False)
print("master (samples x [metadata + genes]):", master.shape)

### 👀 Checkpoint 4: sanity-check the normalisation

**Left:** TPM column sums should all equal 1,000,000. **Right:** per-sample log₂(TPM+1) curves should overlay closely; one curve far off from the rest is an outlier worth investigating.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
ax[0].hist(tpm.sum(), bins=20, color="#4C78A8"); ax[0].set_title("TPM column sums (should be 1e6)")
for s in tpm.columns[:60]:
    ax[1].hist(np.log2(tpm[s] + 1), bins=60, histtype="step", alpha=.4, density=True)
ax[1].set_title("log2(TPM+1) per sample (first 60)")
plt.tight_layout(); plt.show()

---
# Part 5 · The JDM lab's files, key sheet and demographics

🎯 **Goal:** load the JDM lab's processed expression files and find out *who* each sample is, so they can be merged with ours.

💡 **Why this is its own problem:** the lab shipped expression matrices keyed by their own sample IDs. Demographics must be recovered through a separate **key sheet** (`jdm-HC RNASEQ sample ID.csv`) that links their IDs to NIH enrollment IDs and initials, which in turn link to our metadata. Each link can fail, so each is tried in order of reliability.

### 5.1 Load their expression matrices

💡 **Why a `load_lab` function:** both files have a few header rows of sample metadata (RPKM: Cohort, ID; QN: Cohort, MSA, ID) above the gene rows. It splits those off and transposes them so each sample is a row, then re-labels the expression columns with the sample **ID**, so metadata and expression line up by name.

- **`4b_rpkm.csv`**: their RPKM values
- **`4a_log2QN.csv`**: their log2, quantile-normalised values (used in Part 6 to verify the recipe)

In [ ]:
# =============================================================================
# D. the JDM lab's files + key sheet + demographics
# =============================================================================
def load_lab(path, n_meta):
    raw = pd.read_csv(path, low_memory=False)
    g = raw.columns[0]
    meta = raw.iloc[:n_meta].set_index(g).T
    expr = raw.iloc[n_meta:].set_index(g).apply(pd.to_numeric, errors="coerce")
    return meta, expr


m_b, rp = load_lab(JDM_RPKM, 2)                  # rows: Cohort, ID
m_a, qn = load_lab(JDM_QN, 3)                    # rows: Cohort, MSA, ID
m_b.index = m_b["ID"].values
rp.columns = m_b["ID"].values
m_a.index = m_a["ID"].values
qn.columns = m_a["ID"].values
print(f"\nJDM lab RPKM: {rp.shape}, duplicated symbols: {int(rp.index.duplicated().sum())}")
print(m_b["Cohort"].value_counts().to_string())

### 5.2 Read the key sheet and its dates

💡 **Why a custom date parser:** the key sheet mixes **Excel serial numbers** (`40413`) with `m/d/yy` text, and two-digit years need a pivot (≤30 → 2000s). Dates before 2000 or after today are rejected. Also, R silently appended `...2`-style suffixes to duplicate column headers, so `grp` strips those to recover the real diagnosis group.

In [ ]:
KEY_DATE_MIN = pd.Timestamp("2000-01-01")


def parse_key_date(v):
    """Excel serial numbers (40413) and m/d/yy text are both handled."""
    v = str(v).strip()
    if not v:
        return pd.NaT
    if re.fullmatch(r"\d{5}(\.\d+)?", v):
        d = pd.Timestamp("1899-12-30") + pd.Timedelta(days=float(v))
    else:
        m = re.search(r"(\d{1,2})/(\d{1,2})/(\d{2,4})", v)
        if not m:
            return pd.NaT
        mo, da, yr = (int(x) for x in m.groups())
        if yr < 100:
            yr += 2000 if yr <= 30 else 1900
        try:
            d = pd.Timestamp(year=yr, month=mo, day=da)
        except ValueError:
            return pd.NaT
    return d if KEY_DATE_MIN <= d <= DATE_MAX else pd.NaT


key = pd.read_csv(JDM_KEY, dtype=str, keep_default_na=False)
key.columns = [c.strip() for c in key.columns]
key["grp"] = key["Diagnosis"].str.replace(r"\.\.\.\d+$", "", regex=True)   # R added '...2' suffixes
key["_date"] = key["Date"].map(parse_key_date)
print(f"\nkey sheet: {len(key)} rows; dates parsed: {int(key['_date'].notna().sum())}")

### 5.3 Match key-sheet rows to our patient metadata

💡 **Why a multi-step ladder:** no single field is reliable here. The steps run from most to least trustworthy, and the method used is recorded in `matched_by`:

| Step | Method | Trust |
|---|---|---|
| 1 | NIH Enroll ID (plain, or with `R` prefix) | high |
| 2 | Initials interpreted as an ID | high |
| 3 | ID ignoring zero-padding (`JDM0181` = `JDM181`) | high |
| 4 | **Unique** initials, **only if diagnosis doesn't conflict** (HC must look healthy, JDM must look like myositis) | medium |
| 5 | The ID appearing inside the metadata **Notes** field, only if exactly one row contains it and the token is ≥6 characters | weak |

**Age** is then `(key-sheet date − DOB) / 365.25` and the DOB is discarded. Unmatched rows are reported using `shape()` only, so you can see *what format the unmatched IDs have* (e.g. `AAA_999` vs `99-999`) and fix the matching rule without ever printing a real ID.

In [ ]:
def match_key(r):
    """Steps, in order: NIH Enroll ID (plain / R prefix) -> initials as an ID ->
    ID ignoring zero-padding -> unique initials (diagnosis must not conflict) ->
    the ID inside the metadata Notes (weak)."""
    k = normalize_id(r["NIH Enroll ID"])
    ini = str(r["Initials"]).strip()
    ini_up, ini_id = ini.upper(), normalize_id(ini)
    if k and k in meta_lookup:
        return meta_lookup[k], "nih_enroll_id"
    if k.isdigit() and ("R" + k) in meta_lookup:
        return meta_lookup["R" + k], "nih_enroll_id (R prefix)"
    if ini_id and ini_id in meta_lookup:
        return meta_lookup[ini_id], "initials_as_id"
    for tok in (ini_id, k):
        if tok and not tok.isdigit() and zs(tok) in zs_lookup:
            return zs_lookup[zs(tok)], "id_zero_padding"
    i = INI_UNIQUE_ROW.get(ini_up, -1)
    if i >= 0:
        dx = meta_value(i, "dx1")
        ok_dx = (dx == "" or (r["grp"] == "HC" and HEALTHY.search(dx))
                 or (r["grp"] == "JDM" and MYOSITIS.search(dx)))
        return (i, "initials_unique") if ok_dx else (-1, "initials_diagnosis_conflict")
    for tok in (ini_id, k):
        if len(tok) >= 6:
            hits = [j for j, n in notes_norm.items() if n and tok in n]
            if len(hits) == 1:
                return hits[0], "found_in_notes"
    return -1, "none"


mk = key.apply(lambda r: pd.Series(match_key(r), index=["_row", "matched_by"]), axis=1)
key = pd.concat([key, mk], axis=1)
print("\nkey sheet matched to metadata by:")
print(pd.crosstab(key["grp"], key["matched_by"]).to_string())

kdob = pd.to_datetime(key["_row"].map(lambda i: meta_value(i, "dob")), errors="coerce")
key["age_yrs"] = ((key["_date"] - kdob).dt.days / 365.25).round(1)
key.loc[(key["age_yrs"] < 0) | (key["age_yrs"] > 120), "age_yrs"] = np.nan
for col in ("sex", "race", "ethnicity"):
    key[col] = key["_row"].map(lambda i, col=col: meta_value(i, col))
key["patient_key"] = [normalize_id(meta_value(i, "enroll")) or normalize_id(kid)
                      for i, kid in zip(key["_row"], key["NIH Enroll ID"])]
print("age known (key rows):",
      key.groupby("grp")["age_yrs"].apply(lambda s: f"{int(s.notna().sum())}/{len(s)}").to_dict())
un = key[key["_row"] < 0]
print(f"still unmatched: {len(un)}. Formats (masked, no real IDs shown):")
print(pd.DataFrame({"grp": un["grp"], "Initials_format": un["Initials"].map(shape),
                    "NIH_Enroll_ID_format": un["NIH Enroll ID"].map(shape)}
                   ).value_counts().head(12).to_string())
key = key.drop_duplicates("ID").set_index("ID")

### 👀 Checkpoint 5: how were the JDM-lab samples identified?

Strong methods (`nih_enroll_id*`, `initials_as_id`, `id_zero_padding`) are the ones counted as `demographics_trusted` in the merge. Everything else (`initials_unique`, `found_in_notes`, `none`) carries age and sex from a weaker link.

In [ ]:
ct = pd.crosstab(key["grp"], key["matched_by"])
ax = ct.plot.barh(stacked=True, figsize=(11, 3.4))
ax.invert_yaxis(); ax.set_title("JDM-lab key-sheet rows by match method"); ax.set_ylabel("")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

---
# Part 6 · Does our RPKM reproduce theirs?

🎯 **Goal:** before merging two datasets, prove they are on comparable footing, using samples that appear in **both**.

💡 **Why this matters:** if our pipeline and theirs disagree on the *same biological sample*, every cross-dataset difference you later see could be an artifact. Two tests:

**Test 1: our RPKM vs their RPKM, same samples.** Both library-size conventions are tried:

- `all_genes`: divide by reads over *all* genes (what the saved `rnaseq_rpkm.csv` uses).
- `their_gene_set`: divide only by reads inside *their* gene set, since their pipeline likely only counted those genes.

For each shared sample it reports the **Spearman correlation** (are the gene rankings the same?) and the **median ratio** on well-expressed genes (RPKM > 1) (are the *levels* the same?). A ratio near 1.0 under one convention tells you which convention matches their pipeline.

**Test 2: rebuild their QN file from their RPKM file.** If `log2(RPKM+1)` followed by quantile normalisation reproduces their `4a_log2QN.csv`, then we know their recipe and can apply it jointly to both datasets with confidence.

In [ ]:
# ---- compare with our RPKM on the samples both datasets contain -------------
sym_all = pd.Series(gene_names.reindex(counts.index).values, index=counts.index)
in_theirs = sym_all.isin(rp.index)
lib_theirs = counts[in_theirs].sum()


def rpkm_by_symbol(libsize, sample_ids):
    r = counts.loc[ok, sample_ids].div(L[ok], axis=0).div(libsize[sample_ids] / 1e6, axis=1)
    best = r.mean(axis=1).groupby(sym_all[ok].values).idxmax()     # best Ensembl row per symbol
    out = r.loc[best.values]
    out.index = best.index
    return out


pairs = {j: s for j in rp.columns for s in counts.columns if s.endswith("-" + j)}
print(f"\nsamples present in both datasets: {len(pairs)} of {rp.shape[1]}")
if pairs:
    sids = sorted(set(pairs.values()))
    rows = []
    for conv, libsize_ in (("all_genes", counts.sum()), ("their_gene_set", lib_theirs)):
        rr = rpkm_by_symbol(libsize_, sids)
        sg = rr.index.intersection(rp.index)
        for j, s in pairs.items():
            x, y = rr.loc[sg, s], rp.loc[sg, j]
            hi = y > 1
            rows.append((conv, s[:34], round(x.rank().corr(y.rank()), 3),
                         round(float((x[hi] / y[hi]).median()), 2)))
    print(pd.DataFrame(rows, columns=["library_size", "our_sample", "spearman",
                                      "median_ratio_ours_over_theirs"]).to_string(index=False))
    print("share of our reads inside their gene set (median over samples):",
          round(float((lib_theirs / counts.sum()).median()), 3))

common_g = rp.index.intersection(qn.index)
common_s = [c for c in rp.columns if c in qn.columns]
mine = qnorm(np.log2(rp.loc[common_g, common_s] + 1))
cors = [mine[c].corr(qn.loc[common_g, c]) for c in common_s]
print(f"rebuild of their 4a from their 4b via log2(x+1)+QN: median per-sample r = {np.median(cors):.4f}, "
      f"max abs diff = {np.nanmax(np.abs(mine.values - qn.loc[common_g, common_s].values)):.4f}")

### 👀 Checkpoint 6: the comparison, visually

**Left:** our RPKM vs theirs for one shared sample (log-log); points hugging the diagonal mean agreement. **Right:** per-sample correlation of our rebuild of their QN file; every value should be ≈ 1.0.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
if pairs:
    j0, s0 = next(iter(pairs.items()))
    rr = rpkm_by_symbol(lib_theirs, [s0]); sg = rr.index.intersection(rp.index)
    ax[0].scatter(np.log2(rp.loc[sg, j0] + 1), np.log2(rr.loc[sg, s0] + 1), s=3, alpha=.3)
    lim = [0, ax[0].get_xlim()[1]]; ax[0].plot(lim, lim, "r-", lw=.8)
    ax[0].set_xlabel("their log2(RPKM+1)"); ax[0].set_ylabel("our log2(RPKM+1)")
    ax[0].set_title("One shared sample (their_gene_set library)")
else:
    ax[0].text(.5, .5, "no shared samples", ha="center"); ax[0].axis("off")
ax[1].hist(cors, bins=20, color="#59A14F"); ax[1].set_title("QN rebuild: per-sample r vs their 4a")
plt.tight_layout(); plt.show()

---
# Part 7 · Merge both datasets and quantile-normalise jointly

🎯 **Goal:** one genes × samples matrix containing our kept samples *and* the JDM lab's, on a common scale.

💡 **Why each decision:**

- **Genes matched on symbol.** The two datasets use different gene identifiers (Ensembl IDs vs symbols), so symbol is the common key. Where several Ensembl rows share one symbol, the **most-expressed row** is kept.
- **`LIB_CONVENTION`** is applied to *our* RPKM so it is built the same way as theirs (see Part 6).
- **Their samples are excluded in two cases:** (1) the *same sample* is already in our RNA files (it would be counted twice); (2) with `DROP_PEOPLE_ALREADY_IN_OURS`, the *same person* is already in our kept set (repeated measures from one person aren't independent). Every exclusion is written to `merged_excluded_from_jdm_lab.csv`.
- **log2(RPKM+1) then quantile normalisation, jointly.** This reproduces the JDM lab's own recipe (verified in Part 6) and forces every sample, from either dataset, onto the same distribution.
- **Only genes with a value in every sample** are kept, because `qnorm` cannot handle missing values. Samples that are entirely empty are dropped with a warning.

<div class="alert alert-warning">
<b>Quantile normalisation does <i>not</i> remove a dataset batch effect.</b> It aligns distributions, but gene-level shifts between labs survive. That's why a PCA check follows, and why Part 9 does an explicit batch correction.
</div>

### 7.1 Build their sample table

In [ ]:
# =============================================================================
# E. merge both datasets: log2(RPKM+1), joint quantile normalisation
# =============================================================================
kept_ids = list(final_meta["sample_id"])
libsize = lib_theirs if LIB_CONVENTION == "their_gene_set" else counts.sum()
ours_r = rpkm_by_symbol(libsize, kept_ids)

# their samples: metadata from the key sheet, demographics from our metadata sheet
jm = pd.DataFrame(index=rp.columns)
jm["sample_id"] = jm.index
jm["dataset"] = "jdm_lab"
jm["group"] = m_b["Cohort"].reindex(jm.index).values
jm["msa"] = m_a["MSA"].reindex(jm.index).values
jm["pga"] = key["PGA"].reindex(jm.index).values if "PGA" in key.columns else ""
jm["source_file"] = os.path.basename(JDM_RPKM)
for col in ("sex", "race", "ethnicity", "patient_key", "age_yrs"):
    jm[col] = key[col].reindex(jm.index).replace("", np.nan).values
jm["demographics_match"] = key["matched_by"].reindex(jm.index).fillna("not in key sheet").values
jm["demographics_trusted"] = (jm["demographics_match"].str.startswith("nih_enroll_id")
                              | jm["demographics_match"].isin(["initials_as_id", "id_zero_padding"]))
jm["age_date_source"] = np.where(key["_date"].reindex(jm.index).notna().values, "key_sheet_date", "none")
jm["patient_key"] = jm["patient_key"].fillna("")
jm["patient_repeated_within_jdm_lab"] = jm["patient_key"].ne("") & jm["patient_key"].duplicated(keep=False)

### 7.2 Decide which JDM-lab samples to leave out

In [ ]:
# which of their samples to leave out
shared_ids = set(pairs.keys())
ours_people = set(final_meta["patient_key"])
why = pd.Series("", index=jm.index)
why[jm.index.isin(shared_ids)] = "same sample is in our RNA files"
if DROP_PEOPLE_ALREADY_IN_OURS:
    same_person = jm["patient_key"].isin(ours_people) & jm["patient_key"].ne("") & (why == "")
    why[same_person] = "person already in our kept set"
jm_keep = jm[why == ""]
print(f"\nJDM-lab samples: {len(jm)} -> kept {len(jm_keep)}; excluded: {why[why != ''].value_counts().to_dict()}")
pd.DataFrame({"sample_id": jm.index[why != ""], "reason": why[why != ""].values}).to_csv(
    os.path.join(OUT_DIR, "merged_excluded_from_jdm_lab.csv"), index=False)

### 7.3 Build, normalise and save the merged matrix

In [ ]:
# expression: shared genes, log2(RPKM+1), joint quantile normalisation
genes = ours_r.index.intersection(rp.index)
X = pd.concat([ours_r.loc[genes, kept_ids], rp.loc[genes, list(jm_keep.index)]], axis=1)
X = X.replace([np.inf, -np.inf], np.nan)
dead = X.columns[X.isna().all(axis=0)]
if len(dead):
    print("WARNING: samples with no usable values, dropped from the merge:", list(dead))
    X = X.drop(columns=list(dead))
n_before = X.shape[0]
X = X.dropna(axis=0)
print(f"genes with a value in every sample: {X.shape[0]} of {n_before}")
if X.shape[0] == 0:
    raise RuntimeError("No genes left after removing missing values; see the NaN counts above.")
logX = np.log2(X + 1)
QN = qnorm(logX)
print(f"merged matrix: {QN.shape[0]} genes x {QN.shape[1]} samples "
      f"({len(kept_ids)} ours + {len(jm_keep)} JDM lab); library size = {LIB_CONVENTION}")
logX.to_csv(os.path.join(OUT_DIR, "merged_log2_RPKM_before_QN.csv"))
QN.to_csv(os.path.join(OUT_DIR, "merged_log2_RPKM_QN.csv"))

### 7.4 Batch check by PCA, and the merged sample sheet

💡 **Why PCA:** principal components summarise the largest axes of variation. Use the 2,000 most variable genes, and look at where each **dataset** sits. If the two datasets separate along PC1 *within the same group* (e.g. healthy controls from each lab), that gap is a batch effect, not biology. The PC coordinates are saved into `merged_samples.csv` so you can plot or colour them yourself.

In [ ]:
# batch check: PCA on the 2000 most variable genes
top = QN.var(axis=1).sort_values(ascending=False).index[:2000]
Z = QN.loc[top].T.values
Z = Z - Z.mean(axis=0)
U, S, _ = np.linalg.svd(Z, full_matrices=False)
pcs = U[:, :3] * S[:3]
expl = (S ** 2 / (S ** 2).sum())[:3]

ours_sheet = final_meta.assign(dataset="ours", group=final_meta["dx_clean"], msa="", pga="",
                               demographics_match=final_meta["meta_join"])
sheet_cols = ["sample_id", "dataset", "group", "msa", "pga", "patient_key", "sex", "race",
              "ethnicity", "age_yrs", "age_date_source", "demographics_trusted",
              "demographics_match", "source_file"]
merged_meta = pd.concat([ours_sheet[sheet_cols + ["tx_status"]],
                         jm_keep[sheet_cols + ["patient_repeated_within_jdm_lab"]]],
                        ignore_index=True)
merged_meta["tx_status"] = merged_meta["tx_status"].fillna("unknown (JDM lab)")
merged_meta = merged_meta.set_index("sample_id").loc[list(QN.columns)].reset_index()
for i in range(3):
    merged_meta[f"PC{i+1}"] = pcs[:, i]
merged_meta.to_csv(os.path.join(OUT_DIR, "merged_samples.csv"), index=False)
print("PCA variance explained:", [f"{e:.1%}" for e in expl])
print("mean PC1 / PC2 by dataset and group (large gaps between datasets = batch effect):")
print(merged_meta.groupby(["dataset", "group"])[["PC1", "PC2"]].mean().round(2).to_string())
jd = merged_meta[merged_meta["dataset"] == "jdm_lab"]
print(f"JDM-lab samples in the merge with age known: {int(jd['age_yrs'].notna().sum())} of {len(jd)}")

### 👀 Checkpoint 7: PCA before batch correction

**Left:** samples coloured by **dataset**; **right:** by **group**. If the left plot shows two clouds and the right plot's groups line up with them, dataset and biology are tangled. Healthy controls present in *both* datasets are the key evidence: if they don't overlap, there's a batch effect to remove (Part 9).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for d, c in (("ours", "#4C78A8"), ("jdm_lab", "#E15759")):
    s = merged_meta[merged_meta["dataset"] == d]
    ax[0].scatter(s["PC1"], s["PC2"], s=14, alpha=.7, c=c, label=d)
ax[0].legend(); ax[0].set_title("Before batch correction: by dataset")
for g, s in merged_meta.groupby("group"):
    ax[1].scatter(s["PC1"], s["PC2"], s=14, alpha=.7, label=g)
ax[1].legend(fontsize=7, ncol=2); ax[1].set_title("Before batch correction: by group")
for a in ax: a.set_xlabel(f"PC1 ({expl[0]:.0%})"); a.set_ylabel(f"PC2 ({expl[1]:.0%})")
plt.tight_layout(); plt.show()

---
# Part 8 · Processing log

🎯 **Goal:** a Markdown record of exactly what was done, with the numbers filled in **from the run itself**.

💡 **Why generate it instead of writing it:** a hand-written methods note goes stale the moment a setting changes. This log reads the live variables (rules applied, counts, input file sizes and dates, library convention, the QN-rebuild correlation), so it always matches the outputs next to it. It's written **twice**: once early so the file appears in its own output listing, and again after the listing is appended. It also records **known limitations** (approximate ages, weak matches, no treatment labels on JDM-lab samples), which are the caveats a reviewer would ask about.

In [ ]:
# =============================================================================
# F. processing log (written from the run itself, so it cannot go stale)
# =============================================================================
L_ = []
add = L_.append
add("# RNA-seq processing log")
add(f"Generated {datetime.datetime.now():%Y-%m-%d %H:%M}. Python {platform.python_version()}, "
    f"pandas {pd.__version__}, numpy {np.__version__}.\n")
add("## Inputs")
for p in RNA_FILES + [METADATA_PATH] + METADATA_EXTRA + [JDM_RPKM, JDM_QN, JDM_KEY]:
    add(f"- {finfo(p)}")
add(f"- Genes common to our RNA files: {len(counts)} (Ensembl IDs with version; match GENCODE v19)")
add(f"- Gene lengths: union of exon lengths per gene, from {os.path.basename(GTF_URL)}\n")
add("## Our samples: rules applied, in order")
add("0. Samples whose count column is empty or non-numeric are excluded (reason 'no counts in file').")
add(f"1. Whole blood only: Sample Source must be one of {sorted(BLOOD_VALUES)}; names suggesting a cell fraction are excluded.")
add("2. Post-treatment samples excluded (Treatment label not blank and not PRE/baseline).")
add(f"3. For diagnoses matching /{REQUIRE_PRE_FOR_DX}/ a PRE label is required.")
add(f"4. One sample per patient, chosen by: explicit PRE first, then '{KEEP_ONE_PER_PATIENT}', then deepest library.")
add("5. Patient key = metadata Enrollment ID if matched, else the RNA file's enrollment ID, else initials.\n")
add("## Metadata matching (order tried)")
add("enrollment_id; R_prefix_id (RNA '1126' = metadata 'R1126'); initials (unique initials only, and only "
    "if the RNA ID is blank or initials-derived); initials_dx_compat (AGS vs 'Interferon-mediated disease'). "
    "A stated ID absent from the metadata is never overridden by initials. "
    "demographics_trusted = enrollment_id or R_prefix_id matches only.\n")
add("## Counts (our samples)")
add(f"Samples read: {len(rmeta)}")
add("```")
add(pd.crosstab(rmeta["source_file"].str[:45], rmeta["status"]).to_string())
add("```")
add("Metadata match among kept samples:")
add("```")
add(final_meta["meta_join"].value_counts().to_string())
add("```")
add(f"Age known for {int(final_meta['age_yrs'].notna().sum())} of {len(final_meta)} kept samples. "
    f"Age date source: {final_meta['age_date_source'].value_counts().to_dict()}\n")
add("## JDM-lab data and merge")
add(f"- JDM-lab samples: {len(jm)}; excluded {why[why != ''].value_counts().to_dict()}; kept {len(jm_keep)}.")
add("- JDM-lab demographics: key-sheet rows matched to the metadata by NIH Enroll ID, then initials as an ID, "
    "zero-padding-insensitive ID, unique initials (diagnosis-checked), then ID inside Notes (weak, untrusted). "
    f"Matched by: {key['matched_by'].value_counts().to_dict()}.")
add(f"- Samples in both datasets: {len(pairs)}.")
add(f"- Merged matrix: {QN.shape[0]} genes (matched on gene symbol) x {QN.shape[1]} samples.")
add(f"- Our RPKM for the merge uses library size = {LIB_CONVENTION}.")
add("- Normalisation reproduces the JDM lab's own recipe: log2(RPKM+1), then quantile normalisation "
    f"(median per-sample r vs their 4a file = {np.median(cors):.4f}). Applied jointly to both datasets.")
add("- JDM-lab ages use the key-sheet Date (Excel serials and m/d/yy handled), whose meaning "
    "(draw date vs other) has not been confirmed.\n")
add("## Formulas")
add("RPKM = count / (gene_length_kb x library_size_millions). Saved rnaseq_rpkm.csv uses library size = all genes.")
add("TPM = (count/length_kb) / sum(count/length_kb) x 1e6.\n")
add("## Outputs")
open(os.path.join(OUT_DIR, "PROCESSING_LOG.md"), "w").write("\n".join(L_))   # write once so it is listed
for f in sorted(os.listdir(OUT_DIR)):
    fp = os.path.join(OUT_DIR, f)
    if os.path.isfile(fp):
        add(f"- {finfo(fp)}")
add("\n## Known limitations")
add("- Ages for samples without a recorded collection date use the date in the sample name (a sequencing batch stamp): approximate.")
add("- Weak metadata matches are listed in unmatched_or_weak_metadata.csv.")
add("- JDM-lab samples carry no treatment or sample-source labels here, so those filters were not applied to them.")
open(os.path.join(OUT_DIR, "PROCESSING_LOG.md"), "w").write("\n".join(L_))
print("\nwritten to", OUT_DIR, ":")
for f in sorted(os.listdir(OUT_DIR)):
    fp = os.path.join(OUT_DIR, f)
    if os.path.isfile(fp):
        print(f"  {f:46s} {os.path.getsize(fp)/1e6:8.1f} MB")

---
# Part 9 · Batch correction

🎯 **Goal:** remove the **dataset (lab) effect** while **keeping disease-group differences**.

💡 **Why this works, and what it assumes.** A linear model is fitted to every gene:

$$\text{expression} = \beta_0 + \beta_{\text{group}} + \beta_{\text{batch}} \cdot \text{is\_jdm\_lab}$$

Then only the batch term is subtracted. Disease-group terms stay in the model so their signal is **protected** and not absorbed into the "batch" coefficient.

- **HC is the baseline** and is the anchor: healthy controls exist in *both* datasets, so the model can tell a lab shift apart from a disease effect. Without a group present in both, batch and biology cannot be separated at all.
- The `assert` on matrix rank checks that: if a group existed in only one dataset *and* nothing else tied the datasets together, the design would be rank-deficient and the code stops rather than return meaningless coefficients.
- **Assumption:** the lab shift is the *same size for every group*, which is estimated mainly from the HC samples. If the lab shift depends on disease state, this correction is only approximate.

The output is a new file; the uncorrected matrix is left untouched.

In [ ]:
# Batch correction: remove the dataset effect, keep disease-group differences.
# HC exists in both datasets, so it anchors the dataset effect.
import os
import numpy as np, pandas as pd

meta = merged_meta.set_index("sample_id").loc[QN.columns]
grp = meta["group"].astype(str)
G = pd.get_dummies(grp).astype(float).drop(columns="HC")        # HC = baseline
batch = (meta["dataset"] == "jdm_lab").astype(float).values[:, None]
D = np.hstack([np.ones((len(meta), 1)), G.values, batch])
assert np.linalg.matrix_rank(D) == D.shape[1], "design is rank-deficient"

Y = QN.values.T                                                  # samples x genes
beta, *_ = np.linalg.lstsq(D, Y, rcond=None)
corrected = Y - batch @ beta[-1:, :]                             # subtract dataset coefficient

out = pd.DataFrame(corrected.T, index=QN.index, columns=QN.columns)
out.to_csv(os.path.join(OUT_DIR, "merged_log2_RPKM_QN_batchcorrected.csv"))

# did it work? PC1 gap between datasets should shrink a lot
top = pd.Series(corrected.var(axis=0)).nlargest(2000).index
Zc = corrected[:, top] - corrected[:, top].mean(axis=0)
U, S, _ = np.linalg.svd(Zc, full_matrices=False)
pc = U[:, :2] * S[:2]
chk = pd.DataFrame({"dataset": meta["dataset"].values, "group": grp.values,
                    "PC1": pc[:, 0], "PC2": pc[:, 1]})
print("variance explained:", [f"{e:.1%}" for e in (S**2 / (S**2).sum())[:2]])
print(chk.groupby(["dataset", "group"])[["PC1", "PC2"]].mean().round(2).to_string())

### 👀 Checkpoint 9: before vs after

**The test of success:** in the "after" panel, healthy controls from the two datasets should sit on top of each other, while disease groups remain separated from healthy. If datasets still form distinct clouds, the shift isn't a simple additive offset and you'd want a method like ComBat or limma's `removeBatchEffect` with more covariates.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), sharey=False)
mk_ = {"ours": "o", "jdm_lab": "^"}
before = merged_meta.set_index("sample_id").loc[QN.columns]
for a, df_, title in ((ax[0], before.assign(dataset=before["dataset"]), "Before"),
                      (ax[1], chk.set_index(QN.columns), "After")):
    for (d, g), s in df_.groupby(["dataset", "group"]):
        a.scatter(s["PC1"], s["PC2"], s=18, alpha=.7, marker=mk_.get(d, "o"),
                  label=f"{g} ({d})")
    a.set_title(f"{title} batch correction (circle = ours, triangle = JDM lab)")
    a.set_xlabel("PC1"); a.set_ylabel("PC2")
ax[1].legend(fontsize=6, ncol=2, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

---
## ✅ Done

Everything is in `OUT_DIR`. Suggested review order:

1. **Checkpoint 3a**: confirm the pre/post calls. This one regex decides cohort membership.
2. **Checkpoint 6**: shared samples should correlate highly, and the median ratio near 1 tells you which `LIB_CONVENTION` matches the lab.
3. **Checkpoint 9**: healthy controls from both datasets should overlap after correction.
4. Open `PROCESSING_LOG.md` for the run's own account of what was done.
5. Review `unmatched_or_weak_metadata.csv` and fix IDs at the source.